In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('hpp.csv')
df.info(), df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  object 
 2   Description        184508 non-null  object 
 3   Amount(in rupees)  187531 non-null  object 
 4   Price (in rupees)  169866 non-null  float64
 5   location           187531 non-null  object 
 6   Carpet Area        106858 non-null  object 
 7   Status             186916 non-null  object 
 8   Floor              180454 non-null  object 
 9   Transaction        187448 non-null  object 
 10  Furnishing         184634 non-null  object 
 11  facing             117298 non-null  object 
 12  overlooking        106095 non-null  object 
 13  Society            77853 non-null   object 
 14  Bathroom           186703 non-null  object 
 15  Balcony            138596 non-null  object 
 16  Ca

(None,
                Index  Price (in rupees)  Dimensions  Plot Area
 count  187531.000000       1.698660e+05         0.0        0.0
 mean    93765.000000       7.583772e+03         NaN        NaN
 std     54135.681003       2.724171e+04         NaN        NaN
 min         0.000000       0.000000e+00         NaN        NaN
 25%     46882.500000       4.297000e+03         NaN        NaN
 50%     93765.000000       6.034000e+03         NaN        NaN
 75%    140647.500000       9.450000e+03         NaN        NaN
 max    187530.000000       6.700000e+06         NaN        NaN)

In [4]:
df.isnull().sum()

Index                     0
Title                     0
Description            3023
Amount(in rupees)         0
Price (in rupees)     17665
location                  0
Carpet Area           80673
Status                  615
Floor                  7077
Transaction              83
Furnishing             2897
facing                70233
overlooking           81436
Society              109678
Bathroom                828
Balcony               48935
Car Parking          103357
Ownership             65517
Super Area           107685
Dimensions           187531
Plot Area            187531
dtype: int64

In [5]:
# === Clean and preprocess === #
df_clean = df.drop(columns=['Index', 'Title', 'Description', 'Amount(in rupees)','Dimensions', 'Plot Area', 'Super Area', 'Society'])
df_clean = df_clean.dropna(subset=['Price (in rupees)'])
df_clean['Carpet Area'] = df_clean['Carpet Area'].str.extract(r'(\d+)').astype(float)
df_clean['Bathroom'] = pd.to_numeric(df_clean['Bathroom'], errors='coerce')
df_clean['Balcony'] = pd.to_numeric(df_clean['Balcony'], errors='coerce')
df_clean['Floor'] = df_clean['Floor'].str.extract(r'(\d+)').astype(float)
df_clean['Car Parking'] = df_clean['Car Parking'].str.extract(r'(\d+)').astype(float)
df_model = df_clean.dropna(subset=['Carpet Area', 'Bathroom', 'Floor'])

# Fill missing
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = df_model[col].fillna('Unknown')
for col in df_model.select_dtypes(include=['float', 'int']).columns:
    df_model[col] = df_model[col].fillna(0)

# Label encode
le = LabelEncoder()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col])

# === Handle outliers in price === #
q1 = df_model['Price (in rupees)'].quantile(0.25)
q3 = df_model['Price (in rupees)'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
df_model = df_model[(df_model['Price (in rupees)'] >= lower_bound) & (df_model['Price (in rupees)'] <= upper_bound)]

# === Features and target === #
X = df_model.drop(columns=['Price (in rupees)'])
y = df_model['Price (in rupees)']

# Feature scaling for linear models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === Train-test split === #
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [7]:
print(df_model.head())

df_model.to_csv("clean_hpp.csv", index=False)

   Price (in rupees)  location  Carpet Area  Status  Floor  Transaction  \
0             6000.0        67        500.0       0   10.0            3   
1            13799.0        67        473.0       0    3.0            3   
2            17500.0        67        779.0       0   10.0            3   
4            18824.0        67        635.0       0   20.0            3   
6             2538.0        67        550.0       0    4.0            3   

   Furnishing  facing  overlooking  Bathroom  Balcony  Car Parking  Ownership  
0           2       7           17       1.0      2.0          0.0          4  
1           1       0            0       2.0      0.0          1.0          1  
2           2       0            0       2.0      0.0          1.0          1  
4           2       8            1       2.0      0.0          1.0          0  
6           2       7           17       1.0      0.0          0.0          4  


In [14]:
# === Model definitions === #
lin_reg = LinearRegression()
ridge = Ridge()
rf_reg = RandomForestRegressor(random_state=42)
xgb = XGBRegressor(random_state=42)

# === Hyperparameter tuning (example with Random Forest) === #
param_grid_rf = {'n_estimators': [100, 200],'max_depth': [None, 10, 20]}
grid_rf = GridSearchCV(rf_reg, param_grid_rf, cv=5, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_


# === Fit all models === #
lin_reg.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# === Ensemble stacking === #
stack = StackingRegressor(estimators=[('lr', lin_reg),('rf', best_rf),('xgb', xgb)],final_estimator=Ridge())
stack.fit(X_train, y_train)

# === Predictions and evaluation === #
models = {"Linear Regression": lin_reg,"Random Forest (Tuned)": best_rf,"XGBoost": xgb,"Stacked Ensemble": stack}

def evaluate_model(y_true, y_pred, name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Model:")
    print(f"  MSE:  {mse:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.4f}\n")

# Evaluate all
for name, model in models.items():
    y_pred = model.predict(X_test)
    evaluate_model(y_test, y_pred, name)

print('\n')
# === Cross-validation scores === #
def cv_r2(model, name):
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    print(f"{name} CV R²: {scores.mean():.4f} ± {scores.std():.4f}")

cv_r2(lin_reg, "Linear Regression")
cv_r2(best_rf, "Random Forest (Tuned)")
cv_r2(xgb, "XGBoost")
print('\n')

Linear Regression Model:
  MSE:  10741045.59
  RMSE: 3277.35
  R²:   0.1840

Random Forest (Tuned) Model:
  MSE:  1518366.63
  RMSE: 1232.22
  R²:   0.8846

XGBoost Model:
  MSE:  1670825.74
  RMSE: 1292.60
  R²:   0.8731

Stacked Ensemble Model:
  MSE:  1481463.86
  RMSE: 1217.15
  R²:   0.8875



Linear Regression CV R²: 0.1762 ± 0.0130
Random Forest (Tuned) CV R²: 0.8867 ± 0.0032
XGBoost CV R²: 0.8785 ± 0.0023


